# Сегментация зубов. Mask RCNN + Swin Transformer

Инстанс-сегментация зубов на панорамных рентгеновских снимках (ортопантомограммах).
Каждый зуб детектируется и сегментируется как отдельный экземпляр с уникальным классом по нумерации FDI.

## Задача

- **Тип задачи:** Instance Segmentation
- **Объект:** зубы на ортопантомограммах (панорамный рентген)
- **Нумерация:** система FDI (11–18, 21–28, 31–38, 41–48)
- **Количество классов:** 32 (каждый зуб — отдельный класс)
- **Фреймворк:** [Detectron2](https://github.com/facebookresearch/detectron2)

---

## Архитектура модели

### Общая схема

```
Входное изображение
       ↓
  Swin Transformer (backbone)
       ↓
  Feature Pyramid Network (FPN)
       ↓
  Region Proposal Network (RPN)
       ↓
  ROI Heads (bbox + mask)
       ↓
  Предсказания: bbox + маска + класс FDI
```

### Backbone: Swin Transformer Base

| Параметр | Значение |
|---|---|
| Модель | `swin_base_patch4_window7_224` |
| Источник весов | timm (Microsoft, ImageNet-1k pretrain) |
| Лицензия весов | MIT |
| Параметры backbone | ~88M |
| Patch size | 4×4 пикселя |
| Window size | 7×7 патчей |
| Выходные каналы стадий | 128 -> 256 -> 512 -> 1024 |
| Strides стадий | 4 -> 8 -> 16 -> 32 |


**Почему Swin, а не ResNeXt:**
Swin Transformer использует self-attention в скользящих окнах (Shifted Window Attention),
что позволяет модели учитывать глобальный контекст — взаимное расположение зубов в ряду.
Это важно для корректной классификации по FDI: отличить зуб 14 от 15 без понимания
соседних зубов затруднительно.

ResNeXt видит только локальные паттерны через свёртки.

Pretrain на ImageNet-1k (1.3M изображений).     
Если нужен более богатый pretrain — можно использовать `swin_base_patch4_window7_224.ms_in22k`
(ImageNet-21k, 14M изображений), изменив `SWIN_MODEL_NAME` в ноутбуке.    
**Важно**: в данном случае не совсем понятна лицензия (возможность использования в коммерческих целях).

**Gradient Checkpointing:** включён по умолчанию — снижает потребление VRAM примерно вдвое.

### Neck: Feature Pyramid Network (FPN)

Объединяет feature maps всех 4 стадий Swin в единую пирамиду признаков.
Все уровни проецируются в 256 каналов через lateral convolutions.
Выходные уровни: P2, P3, P4, P5, P6 (P6 — MaxPool от P5).

### Head: Mask R-CNN

Стандартный двухэтапный детектор:
1. **RPN** — генерирует proposals (регионы-кандидаты)
2. **ROI Heads** — классифицирует proposals и предсказывает маски

| Параметр | Значение |
|---|---|
| Batch size per image (ROI) | 128 |
| NMS threshold (test) | 0.3 |
| Score threshold (test) | 0.5 |
| Post-NMS proposals (test) | 1000 |
| Post-NMS proposals (train) | 2000 |

### Anchor Generator

| Параметр | Значение |
|---|---|
| Sizes (по уровням FPN) | [32], [64], [128], [256], [512] |
| Aspect ratios | [0.5, 1.0, 2.0] |

---

## Оптимизация

### Оптимизатор: AdamW

AdamW выбран вместо SGD как стандарт для Transformer-based моделей.
Адаптирует learning rate для каждого параметра индивидуально,
weight decay применяется корректно — отдельно от градиентного шага.

| Параметр | Значение |
|---|---|
| Оптимизатор | AdamW |
| Weight decay | 0.05 |
| Weight decay для bias/norm | 0.0 (отключён) |
| Gradient clipping | norm, max_norm=1.0 |

### Learning Rate Schedule

| batch_size | base_lr | warmup_iters |
|---|---|---|
| 2 | 0.0001 | 500 |
| 4 | 0.0002 | 500 |
| 8 | 0.0004 | 1000 |
| 16 | 0.0008 | 1000 |

- **Warmup:** линейный рост от 0 до `base_lr` за `warmup_iters` итераций
- **Decay:** ×0.1 на 60% и 80% от `max_iter` (Step LR)

### Нормализация входа

Модель ожидает RGB изображение, нормализованное по ImageNet статистике:
- Mean: [123.675, 116.280, 103.530]
- Std: [58.395, 57.120, 57.375]

---

## Аугментации

Применяются только на train, реализованы через [Albumentations](https://albumentations.ai/).

| Аугментация | Параметры | Вероятность |
|---|---|---|
| Горизонтальный flip | с зеркальной заменой FDI номеров (1<->2, 3<->4 квадрант) | 0.5 |
| Affine | scale ±5%, translate ±3%, rotate ±3° | 0.5 |
| ElasticTransform | alpha=0.5, sigma=25 | 0.1 |
| CLAHE | clip_limit=2.0, tile_grid=(8,8) | 0.5 |
| RandomBrightnessContrast | ±8% | 0.5 |
| CoarseDropout (чёрный) | 1-2 артефакта, 2-4% размера | 0.05 |
| CoarseDropout (серый) | 1-2 артефакта, 2-4% размера | 0.05 |
| CoarseDropout (белый) | 1-2 артефакта, 2-4% размера | 0.05 |
| GaussNoise | std 0.01-0.04 | 0.2 |

**Горизонтальный flip** реализован отдельно от остальных аугментаций,
потому что требует замены category_id: при флипе левая и правая стороны меняются местами,
поэтому квадрант 1 (зубы 11-18) <-> квадрант 2 (21-28), квадрант 3 (31-38) <-> квадрант 4 (41-48).

---

## Балансировка классов (Class Weights)

В датасете присутствует дисбаланс — зубы мудрости (18, 28, 38, 48) встречаются реже.
Опционально можно включить взвешенный classification loss.

| Метод | Формула | Когда использовать |
|---|---|---|
| `inverse_freq` | weight = 1 / count | сильный дисбаланс |
| `sqrt_inverse_freq` | weight = 1 / √count | умеренный дисбаланс (рекомендуется) |
| `effective_samples` | ENS формула | сложный дисбаланс |

Параметр `class_weight_power` усиливает веса: `final_weight = weight ^ power`.

---

### Руководство по Class Weights для балансировки редких зубов

В датасете зубов есть дисбаланс, что приводит к:
- Модель хуже детектирует зубы мудрости
- Низкий AP для редких классов
- Больше false negatives на зубах мудрости

Class weights увеличивают вес loss для редких классов, заставляя модель уделять им больше внимания.

#### Методы вычисления весов

1. **Inverse Frequency** (сильная балансировка)

`class_weight_method='inverse_freq'`    
Формула: `weight = 1 / count`

Характеристики:
- Максимальное внимание к редким классам
- Сильная балансировка    
**НО**:
- Риск переобучения на редких классах
- Может снизить качество на частых классах

Когда использовать:
- Очень сильный дисбаланс
- Критически важны редкие классы
- Baseline показывает очень плохие результаты

2. **Square Root Inverse Frequency** (умеренная балансировка)

`class_weight_method='sqrt_inverse_freq'`     
Формула: `weight = 1 / sqrt(count)`

Характеристики:
- Умеренная балансировка
- Баланс между частыми и редкими классами
- Меньше риск переобучения
- Хорошо работает в большинстве случаев

Когда использовать:
- Умеренный дисбаланс
- Нужен баланс качества на всех классах

3. **Effective Number of Samples** (адаптивная балансировка)

`class_weight_method='effective_samples'`     
Формула: `weight = (1 - beta) / (1 - beta^count)` где beta = 0.9999

Характеристики:
- Адаптивная балансировка
- Учитывает общее количество образцов
- Теоретически обоснованный метод
- Более сложный

Когда использовать:
- Сложный дисбаланс (разные соотношения)
- Нужен более "умный" подход
- `sqrt_inverse_freq` не дал результата

## Параметр усиления (power)

`class_weight_power=1.5`       
Формула: `final_weight = weight^power`

Когда использовать
- `1.0`	Стандартные веса.	Baseline
- `1.5`	Умеренное усиление. Редкие классы все еще плохо детектируются
- `2.0`	Сильное усиление.	Журтвуем качеством на частых классах

---

## Метрики оценки

### Основные метрики
- **Segmentation AP (mAP50-95)** — основная метрика
- **Segmentation AP50** — IoU threshold 0.5
- **Segmentation AP75** — IoU threshold 0.75
- **BBox AP** — качество детекции

### Дополнительные метрики
- **Dice** — качество сегментации масок
- **IoU** — пересечение над объединением масок
- **Precision / Recall / F1** — по каждому классу FDI
- Вычисляются при IoU@0.5 и IoU@0.75

**Важно:**
- FP приписывается классу предсказания модели,
- FN — классу ground truth.
- TP засчитывается только при совпадении класса и IoU ≥ threshold.


***

## Руководство по размеру изображений
Размер изображений в датасете 640x640, они квадратные.    
При обучении / инференсе есть параметры (например):
```
train_image_sizes=(640,) #(640, 800, 1000, 1200, 1400, 1600), - multi-scale training
test_image_size=640, # Целевой размер inference / Целевой размер valid во время обучения
```

При обучении можно выбрать размер только 640 (как размер изображений в датасете), модель адаптрируется под это размер и при инференсе надо будет указать тот же размер.    
Если мы не знаем размер изображений, который будет приходить на инференс, то можем указать при обучении несколько размеров (multi-scale training), для валидации или инференса указываем один размер. Однако, это ухудшит качество.    
В нашем случае лучше при валидации / инференсе указать 640 и обучать только на 640 - будет наилучшее качество.     
Другой вариант:  
```  
train_image_sizes=(640, 720, 800, 900, 1000, 1200),  # От маленьких до больших
test_image_size=800,  # Средний размер для валидации и inference по умолчанию
```
Или обучаем сначала на 640, затем дообучаем на multi-scale training и увеличиваем test_image_size.      
**Примечание**. В текущем коде по умолчанию изображения прямоугольные и адаптируются под cfg.INPUT.MIN_SIZE_TEST и cfg.INPUT.MAX_SIZE_TEST, в нашем случае (квадратного изображения) они должны совпадать (см. код обучения и инференса: `train_maskrcnn.py -> def setup_config` и `inference_maskrcnn.py -> class MaskRCNNInference`)

---

## Лицензии компонентов

| Компонент | Лицензия | Коммерческое использование |
|---|---|---|
| Detectron2 | Apache 2.0 | ✓ |
| timm | Apache 2.0 | ✓ |
| Swin-B веса (Microsoft) | MIT | ✓ |
| Albumentations | MIT | ✓ |
| PyTorch | BSD | ✓ |



## Установка и импорт библиотек

In [ ]:
#  установка detectron2
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# timm нужен для Swin Transformer backbone
!pip install -q timm

In [ ]:
import os
from google.colab import drive
import zipfile
import json
import random
import torch
import numpy as np
import cv2
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt

# Detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.utils.logger import setup_logger

# Для расчета дополнительных метрик
from detectron2.structures import BoxMode
import pycocotools.mask as mask_util

# настройка визуализации
%matplotlib inline
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = "retina"

In [ ]:
## импортируем скрипты для обучения и инференса модели
RAW_PATH = "https://raw.githubusercontent.com/drSever/MIPT_X-RayDent/refs/heads/master/01_teeth_segmentation/05_MaskRCNN_ST/"

!wget {RAW_PATH + "augmentations.py"} --no-cache
!wget {RAW_PATH + "evaluate_model.py"} --no-cache
!wget {RAW_PATH + "inference_maskrcnn.py"} --no-cache
!wget {RAW_PATH + "swin_backbone.py"} --no-cache
!wget {RAW_PATH + "train_maskrcnn.py"} --no-cache
!wget {RAW_PATH + "visualize_training.py"} --no-cache
!wget {RAW_PATH + "yolo_to_coco_converter.py"} --no-cache

## Для воспроизводимости экспериментов

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # для multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Отключил из-за ошибки памяти Out Of Memory!
    #torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
SEED = 42
set_seed(SEED)

## Получение датасета

In [ ]:
DATASET_PATH_ZIP = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/02_Datasets/Godento2v2.zip'
DATASET_PATH_UNZIP = '/content'
DATASET_DIR = '/content/dataset'

In [ ]:
drive.mount('/content/gdrive')

In [ ]:
# Распаковываем
with zipfile.ZipFile(DATASET_PATH_ZIP, 'r') as zip_ref:
    for file in tqdm(zip_ref.namelist(), desc='Распаковка архива'):
        zip_ref.extract(file, DATASET_PATH_UNZIP)

# переименовываем каталог с распакованным датасетом
os.rename('/content/Godento2v2', '/content/dataset')

print(f"\nАрхив успешно распакован в: {DATASET_DIR}")

## Конвертация разметки YOLO → COCO

In [ ]:
from yolo_to_coco_converter import YOLOtoCOCOConverter

converter = YOLOtoCOCOConverter(
    dataset_path="dataset",
    data_yaml_path="dataset/data.yaml"
)

# Конвертация всех splits
converter.convert_all(output_dir="dataset/coco_annotations")

# Проверка результатов
!ls -la dataset/coco_annotations/

## Обучение модели

In [ ]:
## Параметры

# Пути
DATASET_PATH = "/content/dataset"
COCO_ANNOTATIONS_PATH = "/content/dataset/coco_annotations"
OUTPUT_DIR = "/content/gdrive/MyDrive/Colab_Notebooks/Startup/04_TeethSegmentation/MaskRCNN_det2_swin"

# Модель
NUM_CLASSES = 32    # фон для instance segmentation не нужен
#RESUME_FROM = None  # Путь к чекпоинту для продолжения обучения (или None)
RESUME_FROM = f"{OUTPUT_DIR}/model_best.pth"

# Swin Transformer backbone
# Варианты (по возрастанию качества и требований к VRAM):
#   swin_tiny_patch4_window7_224   — ~28M params, ~8GB  VRAM
#   swin_small_patch4_window7_224  — ~50M params, ~10GB VRAM
#   swin_base_patch4_window7_224   — ~88M params, ~14GB VRAM
#   swin_large_patch4_window7_224  — ~197M params, ~24GB VRAM
SWIN_MODEL_NAME = "swin_base_patch4_window7_224"
SWIN_PRETRAINED = True  # ImageNet-21k pretrained веса через timm

# Параметры обучения
MAX_ITER = 10000          # Количество итераций (при дообучении = + сколько надо дообучить)
BATCH_SIZE = 16            # Размер батча
BASE_LR = 0.0008          # LR для Swin (ниже чем для ResNet/ResNeXt)
WARMUP_ITERS = 1000        # Warmup итераций, обычно 5% от max_iter или 500-1000 итераций
LR_STEPS = None           # Шаги снижения LR, кортеж (None = авто: 60% и 80% от MAX_ITER) или явно (12000, 16000)

# Оценка и сохранение
EVAL_PERIOD = 200         # Оценка на валидации каждые N итераций
CHECKPOINT_PERIOD = 5000  # Сохранение чекпоинтов каждые N итераций

# Аугментации
USE_AUGMENTATIONS = True  # True = использовать аугментации из augmentations.py, False = без аугментаций

# Class Weights (балансировка редких зубов)
CLASS_WEIGHTS = True  # False = выключено, True = включить веса классов
CLASS_WEIGHT_METHOD = 'sqrt_inverse_freq'  # Метод: 'inverse_freq', 'sqrt_inverse_freq', 'effective_samples'
CLASS_WEIGHT_POWER = 1.0  # Степень усиления (1.0 = без усиления, 1.5 = умеренное, 2.0 = сильное)

# Размер изображений
TRAIN_IMAGE_SIZES = (640,)    # или (640, 800, 1000, 1200, 1400, 1600) - multi-scale training
TEST_IMAGE_SIZE = 640     # Целевой размер inference / Целевой размер valid во время обучения

In [ ]:
%%time
## Обучение

from train_maskrcnn import train_model



cfg, trainer, history = train_model(
    dataset_path=DATASET_PATH,
    coco_annotations_path=COCO_ANNOTATIONS_PATH,
    output_dir=OUTPUT_DIR,
    resume_from=RESUME_FROM,
    num_classes=NUM_CLASSES,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
    base_lr=BASE_LR,
    warmup_iters=WARMUP_ITERS,
    lr_steps=LR_STEPS,
    eval_period=EVAL_PERIOD,
    checkpoint_period=CHECKPOINT_PERIOD,
    use_augmentations=USE_AUGMENTATIONS,
    class_weights=CLASS_WEIGHTS,
    class_weight_method=CLASS_WEIGHT_METHOD,
    class_weight_power=CLASS_WEIGHT_POWER,
    train_image_sizes=TRAIN_IMAGE_SIZES,
    test_image_size=TEST_IMAGE_SIZE,
    swin_model_name=SWIN_MODEL_NAME,
    swin_pretrained=SWIN_PRETRAINED,
)

## Визуализация результатов обучения

In [ ]:
from visualize_training import TrainingVisualizer

# Создаем визуализатор
visualizer = TrainingVisualizer(f"{OUTPUT_DIR}/training_history.json")

# Выводим сводку
visualizer.print_summary()

# Строим все графики
visualizer.plot_all(output_dir=f"{OUTPUT_DIR}/plots")

# Или по отдельности:
# visualizer.plot_losses()
# visualizer.plot_learning_rate()
# visualizer.plot_validation_metrics()
# visualizer.plot_train_val_comparison()

## Оценка на TEST

In [ ]:
from evaluate_model import evaluate_model

results = evaluate_model(
    # путь к модели
    model_path=f"{OUTPUT_DIR}/model_best.pth",
    # путь к датасету
    dataset_path=DATASET_PATH,
    # путь к аннотациям COCO
    coco_annotations_path=COCO_ANNOTATIONS_PATH,
    # куда сохранять результаты
    output_dir=f"{OUTPUT_DIR}/evaluation_results",
    # число классов
    num_classes=NUM_CLASSES,
    # размер (минимальный, если не квадрат) изображения на вход модели
    test_image_size=TEST_IMAGE_SIZE,
    # порог уверенности
    score_threshold=0.5,
    # тестовая подвыборка
    dataset_name="test"
)

## Инференс

In [ ]:
from inference_maskrcnn import MaskRCNNInference

inference = MaskRCNNInference(
    model_path=f"{OUTPUT_DIR}/model_best.pth",
    num_classes=NUM_CLASSES,
    score_threshold=0.5,             # выставляем для всех изображений по умолчанию
    test_image_size=TEST_IMAGE_SIZE  # Должен совпадать с обучением!
)

outputs, vis = inference.predict_and_visualize(
    "/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest/№3.JPG",
    score_threshold=0.9 # выставляем для данного изображения
)

# Информация о предсказаниях
info = inference.get_predictions_info(outputs)
print(f"\nНайдено зубов: {info['num_instances']}")
print(f"FDI номера: {info['fdi_numbers']}")
print(f"Уверенность: {[f'{s:.3f}' for s in info['scores']]}")


In [ ]:
# инференс пакета изображений
results = inference.predict_batch(
    image_dir="/content/gdrive/MyDrive/Colab_Notebooks/Startup/01_Data/RealTest",
    output_dir="/content/gdrive/MyDrive/Colab_Notebooks/Startup/04_TeethSegmentation/MaskRCNN_det2_swin/inference_real",
    score_threshold=0.9
)

In [ ]:
!pip list --format=freeze > requirements.txt